# LightAD — Log Anomaly Detection Benchmark

**Paper:** Yu et al. (ICSE 2024) — *Deep Learning or Classical Machine Learning? An Empirical Study on Log-Based Anomaly Detection*  
**Models:** KNN · Decision Tree · SLFN (MLP)  
**Datasets:** HDFS_v1 · BGL  

---
### Kaggle Setup
1. Upload dataset files to Kaggle Dataset:
   - `HDFS.log` → `/kaggle/input/logad-data/hdfs/HDFS.log`
   - `anomaly_label.csv` → `/kaggle/input/logad-data/hdfs/anomaly_label.csv`
   - `BGL.log` → `/kaggle/input/logad-data/bgl/BGL.log`
2. Add this notebook's dataset
3. Enable GPU (optional — LightAD runs fine on CPU)
4. Run All

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q scikit-learn bayesian-optimization distance

In [ ]:
import os, sys, time, json, random
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn import tree
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_recall_fscore_support, average_precision_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Kaggle paths — adjust if running locally
HDFS_LOG   = '/kaggle/input/logad-data/hdfs/HDFS.log'
HDFS_LABEL = '/kaggle/input/logad-data/hdfs/anomaly_label.csv'
BGL_LOG    = '/kaggle/input/logad-data/bgl/BGL.log'
OUTPUT_DIR = '/kaggle/working/lightad_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths OK')

## 1. Data Loading & Preprocessing

In [ ]:
# ── HDFS preprocessing (from LightAD source, adapted) ───────────────────────
def preprocess_hdfs(log_path, label_path, n_shuffles=5, train_ratio=0.8):
    """Parse HDFS.log → vectorized block sequences with labels."""
    print('Loading HDFS labels...')
    label_df = pd.read_csv(label_path)
    label_dict = {}
    for _, row in label_df.iterrows():
        label_dict[row['BlockId']] = '-' if row['Label'] == 'Normal' else '+'

    print(f'Loading HDFS log ({log_path})...')
    total_data = {}   # block_id -> list of token sequences
    data_vocab  = []  # all unique token sequences (vocabulary)

    with open(log_path, 'rb') as f:
        for i, raw_line in enumerate(f):
            if i % 1_000_000 == 0:
                print(f'  {i:,} lines processed, vocab size: {len(data_vocab)}')
            try:
                line = raw_line.decode('utf-8')[:-1].split(' ')
            except:
                continue
            content = line[5:]
            # find block ID
            current_blk = None
            tokens = []
            for c in content:
                if c.startswith('blk_') and current_blk is None:
                    current_blk = c.strip('.')
                # keep only alphabetical tokens (no digits)
                if all(not ch.isdigit() for ch in c):
                    tokens.append(c)
            if current_blk is None:
                continue
            if current_blk not in total_data:
                total_data[current_blk] = []
            total_data[current_blk].append(tokens)
            if tokens not in data_vocab:
                data_vocab.append(tokens)

    print(f'Vocab size: {len(data_vocab)}, Blocks: {len(total_data)}')

    # Vectorize: count-of-event-type vector per block
    def vectorize(seq_list):
        vec = [0] * len(data_vocab)
        for seq in seq_list:
            if seq in data_vocab:
                vec[data_vocab.index(seq)] += 1
        return vec

    blocks = list(total_data.keys())
    X = [vectorize(total_data[b]) for b in blocks]
    y = [label_dict.get(b, '-') for b in blocks]

    # Create n_shuffles train/test splits
    splits = []
    for _ in range(n_shuffles):
        x_tr, x_te, y_tr, y_te = train_test_split(
            X, y, test_size=1-train_ratio, random_state=random.randint(0, 9999)
        )
        splits.append((x_tr, x_te, y_tr, y_te))

    print('HDFS preprocessing done.')
    return splits


hdfs_splits = preprocess_hdfs(HDFS_LOG, HDFS_LABEL, n_shuffles=5)
print(f'Train size: {len(hdfs_splits[0][0])}, Test size: {len(hdfs_splits[0][1])}')

In [ ]:
# ── BGL preprocessing ────────────────────────────────────────────────────────
def preprocess_bgl(log_path, train_ratio=0.8, max_lines=None):
    """Parse BGL.log using semantic token matching (no vectorization)."""
    print(f'Loading BGL log...')
    all_seqs, all_labels = [], []
    error_types = set()

    with open(log_path, 'rb') as f:
        for i, raw_line in enumerate(f):
            if max_lines and i >= max_lines:
                break
            if i % 1_000_000 == 0 and i > 0:
                print(f'  {i:,} lines')
            try:
                line = raw_line.decode('utf-8')[:-1]
            except:
                continue
            parts = line.split(' ')
            if len(parts) < 9:
                continue
            label  = parts[0]           # '-' = normal, else error type
            tokens = parts[8:]          # log message tokens
            # keep only alphabetical tokens
            clean = [t for t in tokens if all(not c.isdigit() for c in t) and t]
            if label != '-':
                error_types.add(label)
            all_seqs.append(clean)
            all_labels.append(label)

    n_train = int(train_ratio * len(all_seqs))
    x_train = all_seqs[:n_train]
    y_train = all_labels[:n_train]
    x_test  = all_seqs[n_train:]
    y_test  = all_labels[n_train:]

    anom_rate = sum(1 for l in all_labels if l != '-') / len(all_labels)
    print(f'BGL: {len(all_seqs):,} lines | anomaly rate: {anom_rate:.4f}')
    print(f'Train: {len(x_train):,} | Test: {len(x_test):,}')
    return x_train, x_test, y_train, y_test


bgl_x_train, bgl_x_test, bgl_y_train, bgl_y_test = preprocess_bgl(BGL_LOG)

## 2. Models

In [ ]:
def compute_metrics(y_true, y_pred):
    """Compute P/R/F1/Specificity/Balanced Acc using LightAD's '+'/'-' convention."""
    tp = fp = fn = tn = 0
    for t, p in zip(y_true, y_pred):
        if   p == '-' and t == '-': tn += 1
        elif p == '-' and t != '-': fn += 1
        elif p != '-' and t == '-': fp += 1
        else:                       tp += 1
    prec  = tp / (tp + fp) if (tp + fp) else 0
    rec   = tp / (tp + fn) if (tp + fn) else 0
    f1    = 2*prec*rec / (prec+rec) if (prec+rec) else 0
    spec  = tn / (fp + tn) if (fp + tn) else 0
    bal   = (spec + rec) / 2
    return dict(precision=prec, recall=rec, f1=f1, specificity=spec,
                balanced_acc=bal, tp=tp, fp=fp, fn=fn, tn=tn)


def run_knn(x_train, x_test, y_train, n_neighbors=1):
    # deduplicate training set
    seen, x_tr_d, y_tr_d = set(), [], []
    for x, y in zip(x_train, y_train):
        key = str(x)
        if key not in seen:
            seen.add(key)
            x_tr_d.append(x)
            y_tr_d.append(y)
    t0 = time.time()
    clf = KNeighborsClassifier(n_neighbors=n_neighbors)
    clf.fit(x_tr_d, y_tr_d)
    train_t = time.time() - t0
    t0 = time.time()
    # cache predictions for duplicate sequences
    cache, preds = {}, []
    for x in x_test:
        key = str(x)
        if key not in cache:
            cache[key] = clf.predict([x])[0]
        preds.append(cache[key])
    infer_t = time.time() - t0
    return preds, train_t, infer_t


def run_dt(x_train, x_test, y_train):
    t0 = time.time()
    clf = tree.DecisionTreeClassifier()
    clf.fit(x_train, y_train)
    train_t = time.time() - t0
    t0 = time.time()
    preds = list(clf.predict(x_test))
    infer_t = time.time() - t0
    return preds, train_t, infer_t


def run_slfn(x_train, x_test, y_train, hidden=(25,)):
    t0 = time.time()
    clf = MLPClassifier(hidden_layer_sizes=hidden, max_iter=500, random_state=SEED)
    clf.fit(x_train, y_train)
    train_t = time.time() - t0
    t0 = time.time()
    preds = list(clf.predict(x_test))
    infer_t = time.time() - t0
    return preds, train_t, infer_t


def jaccard_distance(a, b):
    sa, sb = set(a), set(b)
    return 1 - len(sa & sb) / len(sa | sb) if (sa | sb) else 1.0


def run_bgl_knn(x_train, x_test, y_train, sample_ratio=0.1):
    """BGL: semantic match (Jaccard) on 10% sample, window evaluation."""
    idx = np.random.choice(len(x_train), int(sample_ratio * len(x_train)), replace=False)
    xs, ys = [x_train[i] for i in idx], [y_train[i] for i in idx]
    t0 = time.time()
    preds = []
    for i, x in enumerate(x_test):
        if i % 100_000 == 0:
            print(f'  BGL testing {i:,}/{len(x_test):,}')
        # find nearest neighbor by Jaccard distance
        best_label, best_dist = '-', float('inf')
        for xtr, ytr in zip(xs, ys):
            d = jaccard_distance(x, xtr)
            if d < best_dist:
                best_dist, best_label = d, ytr
        preds.append(best_label)
    infer_t = time.time() - t0
    return preds, 0.0, infer_t

print('Model functions defined.')

## 3. Train & Evaluate — HDFS

In [ ]:
hdfs_results = {}

for model_name, run_fn in [('knn', run_knn), ('dt', run_dt), ('slfn', run_slfn)]:
    p_all, r_all, f_all, s_all, b_all = [], [], [], [], []
    tt_all, ti_all = [], []

    for fold, (x_tr, x_te, y_tr, y_te) in enumerate(hdfs_splits):
        preds, train_t, infer_t = run_fn(x_tr, x_te, y_tr)
        m = compute_metrics(y_te, preds)
        p_all.append(m['precision']); r_all.append(m['recall'])
        f_all.append(m['f1']); s_all.append(m['specificity'])
        b_all.append(m['balanced_acc'])
        tt_all.append(train_t); ti_all.append(infer_t)
        print(f'  [{model_name.upper()} fold {fold+1}] '
              f'P={m["precision"]:.4f} R={m["recall"]:.4f} F1={m["f1"]:.4f}')

    hdfs_results[model_name] = {
        'precision':    np.mean(p_all),
        'recall':       np.mean(r_all),
        'f1':           np.mean(f_all),
        'specificity':  np.mean(s_all),
        'balanced_acc': np.mean(b_all),
        'train_time_s': np.mean(tt_all),
        'infer_time_s': np.mean(ti_all),
    }
    print(f'=> HDFS {model_name.upper()} avg F1: {np.mean(f_all):.4f}\n')

print('\n=== HDFS SUMMARY ===')
for m, v in hdfs_results.items():
    print(f'{m.upper():6s}  P={v["precision"]:.4f}  R={v["recall"]:.4f}  '
          f'F1={v["f1"]:.4f}  train={v["train_time_s"]:.2f}s  infer={v["infer_time_s"]:.2f}s')

## 4. Train & Evaluate — BGL

In [ ]:
print('Running BGL KNN (semantic match, 10% training sample)...')
bgl_preds, bgl_tt, bgl_ti = run_bgl_knn(bgl_x_train, bgl_x_test, bgl_y_train, sample_ratio=0.1)
bgl_m = compute_metrics(bgl_y_test, bgl_preds)

bgl_results = {'knn': {**bgl_m, 'train_time_s': bgl_tt, 'infer_time_s': bgl_ti}}
print(f'\n=== BGL SUMMARY ===')
print(f'KNN  P={bgl_m["precision"]:.4f}  R={bgl_m["recall"]:.4f}  '
      f'F1={bgl_m["f1"]:.4f}  infer={bgl_ti:.1f}s')

## 5. Save Results

In [ ]:
all_results = {
    'metadata': {'model': 'LightAD', 'paper': 'ICSE 2024', 'seed': SEED},
    'hdfs': hdfs_results,
    'bgl':  bgl_results,
}
out_path = os.path.join(OUTPUT_DIR, 'lightad_results.json')
with open(out_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Saved: {out_path}')

## 6. Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
metrics = ['precision', 'recall', 'f1']
x = np.arange(len(metrics))
width = 0.25
colors = ['#2196F3', '#4CAF50', '#FF9800']

# HDFS
ax = axes[0]
for i, (model_name, vals) in enumerate(hdfs_results.items()):
    bars = [vals[m] for m in metrics]
    ax.bar(x + i*width, bars, width, label=model_name.upper(), color=colors[i], alpha=0.85)
ax.set_title('LightAD — HDFS_v1', fontsize=12, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(['Precision', 'Recall', 'F1'])
ax.set_ylim(0.98, 1.001)
ax.legend()
ax.grid(axis='y', alpha=0.3)
for i, (model_name, vals) in enumerate(hdfs_results.items()):
    for j, m in enumerate(metrics):
        ax.text(j + i*width, vals[m] + 0.0001, f'{vals[m]:.4f}',
                ha='center', va='bottom', fontsize=7)

# BGL
ax = axes[1]
bgl_bars = [bgl_m[m] for m in metrics]
ax.bar(x, bgl_bars, 0.5, label='KNN', color=colors[0], alpha=0.85)
ax.set_title('LightAD — BGL', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Precision', 'Recall', 'F1'])
ax.set_ylim(0.95, 1.01)
ax.legend()
ax.grid(axis='y', alpha=0.3)
for j, m in enumerate(metrics):
    ax.text(j, bgl_m[m] + 0.001, f'{bgl_m[m]:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lightad_results.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Runtime comparison
fig, ax = plt.subplots(figsize=(8, 4))
models = list(hdfs_results.keys())
train_times = [hdfs_results[m]['train_time_s'] for m in models]
infer_times = [hdfs_results[m]['infer_time_s'] for m in models]
x = np.arange(len(models))
ax.bar(x - 0.2, train_times, 0.4, label='Train', color='#2196F3', alpha=0.8)
ax.bar(x + 0.2, infer_times, 0.4, label='Inference', color='#FF9800', alpha=0.8)
ax.set_title('LightAD — Runtime (HDFS)', fontsize=12, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels([m.upper() for m in models])
ax.set_ylabel('Time (seconds)'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lightad_runtime.png'), bbox_inches='tight')
plt.show()
print('Done!')